In [1]:
# Cell 1 (Cập nhật huggingface_hub để sửa lỗi parse_hf_uri):
!pip install -q --no-deps qwen-vl-utils
!pip install -q -U huggingface_hub
!pip install -q "transformers>=4.45.0" "git+https://github.com/huggingface/diffusers.git" accelerate opencv-python bitsandbytes sentencepiece tiktoken

print("✅ Đã hoàn tất nâng cấp huggingface_hub!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 774.9/774.9 kB 12.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 82.4 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 48.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 32.8 MB/s eta 0:00:00
✅ Đã hoàn tất nâng cấp huggingface_hub!


In [2]:
# Cell Helper: Định nghĩa hàm in báo cáo sạch UI
import os, sys, re, time, torch, cv2, numpy as np
from pathlib import Path

TASK_MAPPING = {
    0: "Assembling the spring", 1: "Placing white plastic", 2: "Screwing-1",
    3: "Inflating valve", 4: "Placing black plastic", 5: "Screwing-2", 6: "Fixing cable"
}

def parse_ground_truth(file_path: str):
    name = Path(file_path).name.lower()
    parent = Path(file_path).parent.name.lower()
    m = re.search(r'task_?0*([0-6])\b', name) or re.search(r'task_?0*([0-6])\b', parent)
    if m: return int(m.group(1))
    m7 = re.search(r'task_?0*([1-7])\b', name) or re.search(r'task_?0*([1-7])\b', parent)
    if m7: return max(0, int(m7.group(1)) - 1)
    return None

def parse_predicted_task(text_output: str):
    text_lower = text_output.lower()
    for name, task_id in [("assembling the spring", 0), ("placing white plastic", 1), ("screwing-1", 2), ("inflating valve", 3), ("placing black plastic", 4), ("screwing-2", 5), ("fixing cable", 6)]:
        if name in text_lower: return task_id
    num_match = re.search(r'(?:task|class|answer)\s*[:#-]?\s*([0-6])\b', text_lower)
    if num_match: return int(num_match.group(1))
    return None

def get_hatrec_video_files():
    candidate_paths = [
        "/kaggle/input/datasets/ayoznur/hatrec-video-dataset",
        "/kaggle/input/hatrec-video-dataset",
        "/kaggle/input/real-world-industrial-assembly-action-dataset",
        "/kaggle/input/datasets/ayoznur/real-world-industrial-assembly-action-dataset",
        "/kaggle/working/mini_cosmos/videos",
        "./videos"
    ]
    for cand in candidate_paths:
        if os.path.exists(cand):
            vids = sorted(list(Path(cand).rglob("*.mp4")) + list(Path(cand).rglob("*.avi")))
            if len(vids) > 0:
                return vids[:546]
    return []

def create_static_frame_video(video_path: str, temp_output_path: str = "/tmp/static_test.mp4", num_frames: int = 16):
    cap = cv2.VideoCapture(str(video_path))
    ret, first_frame = cap.read()
    cap.release()
    if not ret or first_frame is None: return None
    h, w, c = first_frame.shape
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(temp_output_path, fourcc, 1.0, (w, h))
    for _ in range(num_frames): out.write(first_frame)
    out.release()
    return temp_output_path

def print_clean_summary(model_name: str, mode_name: str, total_eval: int, correct_count: int, elapsed_sec: float, peak_vram_gb: float):
    acc = (correct_count / total_eval * 100.0) if total_eval > 0 else 0.0
    print("\n" + "="*75)
    print(f"📊 BÁO CÁO KẾT QUẢ ĐÁNH GIÁ CHÍNH THỨC:")
    print(f"   • Mô hình (Model)              : {model_name}")
    print(f"   • Chế độ chạy (Test Mode)      : {mode_name}")
    print(f"   • Số câu dự đoán (Evaluated)   : {total_eval} videos")
    print(f"   • Số câu trả lời ĐÚNG         : {correct_count} videos")
    print(f"   • Độ chính xác (Accuracy)       : {acc:.2f}%")
    print(f"   • Thời gian thực thi (Elapsed) : {elapsed_sec:.2f} giây ({elapsed_sec/60.0:.2f} phút)")
    print(f"   • Peak GPU VRAM Tiêu Thụ        : {peak_vram_gb:.2f} GB VRAM")
    print("="*75 + "\n")
    

In [3]:
# Cell 2: Test Qwen2-VL-2B (Mode Bình Thường - Standalone)
import torch, time, os
from pathlib import Path
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

video_files = get_hatrec_video_files()
if not video_files:
    print("❌ KHÔNG TÌM THẤY VIDEO!")
else:
    model_id = "Qwen/Qwen2-VL-2B-Instruct"
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    start_t = time.time()

    processor = AutoProcessor.from_pretrained(model_id)
    model = Qwen2VLForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16, device_map="cuda:0")
    model.eval()

    prompt_text = "Watch this video carefully and classify the exact assembly action being performed into one of 7 choices: Task 0: Assembling the spring, Task 1: Placing white plastic, Task 2: Screwing-1, Task 3: Inflating valve, Task 4: Placing black plastic, Task 5: Screwing-2, Task 6: Fixing cable. State choice as: 'Task X: [Task Name]'"

    correct_count, total_eval = 0, 0
    for video_file in video_files:
        gt_task = parse_ground_truth(str(video_file))
        messages = [{"role": "user", "content": [{"type": "video", "video": str(video_file), "max_pixels": 360*420, "fps": 1.0}, {"type": "text", "text": prompt_text}]}]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to("cuda:0")
        
        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=32, do_sample=False)
        generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
        output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True)[0]
        
        pred_task = parse_predicted_task(output_text)
        if gt_task is not None and pred_task == gt_task: correct_count += 1
        total_eval += 1
        torch.cuda.empty_cache()

    elapsed = time.time() - start_t
    peak_vram = torch.cuda.max_memory_allocated() / (1024**3)
    print_clean_summary("Qwen2-VL-2B-Instruct", "Dynamic Native Video (Bình thường)", total_eval, correct_count, elapsed, peak_vram)


preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

qwen-vl-utils using torchcodec to read video.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



📊 BÁO CÁO KẾT QUẢ ĐÁNH GIÁ CHÍNH THỨC:
   • Mô hình (Model)              : Qwen2-VL-2B-Instruct
   • Chế độ chạy (Test Mode)      : Dynamic Native Video (Bình thường)
   • Số câu dự đoán (Evaluated)   : 546 videos
   • Số câu trả lời ĐÚNG         : 98 videos
   • Độ chính xác (Accuracy)       : 17.95%
   • Thời gian thực thi (Elapsed) : 493.13 giây (8.22 phút)
   • Peak GPU VRAM Tiêu Thụ        : 4.23 GB VRAM



In [4]:
# Cell 3: Test Qwen2-VL-2B (Mode Đóng Băng Frame 16x - Standalone)
import torch, time, os
from pathlib import Path
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

video_files = get_hatrec_video_files()
if not video_files:
    print("❌ KHÔNG TÌM THẤY VIDEO!")
else:
    model_id = "Qwen/Qwen2-VL-2B-Instruct"
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    start_t = time.time()

    processor = AutoProcessor.from_pretrained(model_id)
    model = Qwen2VLForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16, device_map="cuda:0")
    model.eval()

    prompt_text = "Watch this video carefully and classify the exact assembly action being performed into one of 7 choices: Task 0: Assembling the spring, Task 1: Placing white plastic, Task 2: Screwing-1, Task 3: Inflating valve, Task 4: Placing black plastic, Task 5: Screwing-2, Task 6: Fixing cable. State choice as: 'Task X: [Task Name]'"

    temp_static_video = "/tmp/static_test_2b.mp4"
    correct_count, total_eval = 0, 0

    for video_file in video_files:
        gt_task = parse_ground_truth(str(video_file))
        eval_video = create_static_frame_video(str(video_file), temp_output_path=temp_static_video, num_frames=16)
        if not eval_video: continue
        
        messages = [{"role": "user", "content": [{"type": "video", "video": eval_video, "max_pixels": 360*420, "fps": 1.0}, {"type": "text", "text": prompt_text}]}]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to("cuda:0")
        
        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=32, do_sample=False)
        generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
        output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True)[0]
        
        pred_task = parse_predicted_task(output_text)
        if gt_task is not None and pred_task == gt_task: correct_count += 1
        total_eval += 1
        torch.cuda.empty_cache()

    elapsed = time.time() - start_t
    peak_vram = torch.cuda.max_memory_allocated() / (1024**3)
    print_clean_summary("Qwen2-VL-2B-Instruct", "Static-Frame Test (Đóng Băng Frame)", total_eval, correct_count, elapsed, peak_vram)


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]


📊 BÁO CÁO KẾT QUẢ ĐÁNH GIÁ CHÍNH THỨC:
   • Mô hình (Model)              : Qwen2-VL-2B-Instruct
   • Chế độ chạy (Test Mode)      : Static-Frame Test (Đóng Băng Frame)
   • Số câu dự đoán (Evaluated)   : 546 videos
   • Số câu trả lời ĐÚNG         : 105 videos
   • Độ chính xác (Accuracy)       : 19.23%
   • Thời gian thực thi (Elapsed) : 1403.31 giây (23.39 phút)
   • Peak GPU VRAM Tiêu Thụ        : 8.25 GB VRAM



In [3]:
# Cell 4: Test Qwen2-VL-7B (Tối ưu xả bộ nhớ C++ CUDA Allocator)
import torch, time, os, gc
from pathlib import Path
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

# 1. XẢ TRIỆT ĐỂ BỘ NHỚ C++ PYTORCH ALLOCATOR
for var_name in list(globals().keys()):
    if 'model' in var_name or 'processor' in var_name:
        try: del globals()[var_name]
        except: pass

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect() # Xả C++ IPC Memory Pool

video_files = get_hatrec_video_files()
if not video_files:
    print("❌ KHÔNG TÌM THẤY VIDEO!")
else:
    model_id_7b = "Qwen/Qwen2-VL-7B-Instruct"
    torch.cuda.reset_peak_memory_stats()
    start_t = time.time()

    # 2. Giới hạn hạn mức VRAM tiêu thụ tối đa 7.5GB cho GPU 0 (phần còn lại đẩy tạm sang RAM CPU)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        llm_int8_enable_fp32_cpu_offload=True
    )
    
    max_mem = {0: "7.5GiB", "cpu": "20GiB"}

    processor_7b = AutoProcessor.from_pretrained(model_id_7b)
    model_7b = Qwen2VLForConditionalGeneration.from_pretrained(
        model_id_7b,
        quantization_config=bnb_config,
        device_map="auto",
        max_memory=max_mem,
        offload_folder="/tmp/offload_7b",
        low_cpu_mem_usage=True
    )
    model_7b.eval()

    prompt_text = "Watch this video carefully and classify the exact assembly action being performed into one of 7 choices: Task 0: Assembling the spring, Task 1: Placing white plastic, Task 2: Screwing-1, Task 3: Inflating valve, Task 4: Placing black plastic, Task 5: Screwing-2, Task 6: Fixing cable. State choice as: 'Task X: [Task Name]'"

    correct_count, total_eval = 0, 0
    for video_file in video_files:
        gt_task = parse_ground_truth(str(video_file))
        messages = [{"role": "user", "content": [{"type": "video", "video": str(video_file), "max_pixels": 360*420, "fps": 1.0}, {"type": "text", "text": prompt_text}]}]
        text = processor_7b.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor_7b(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to(model_7b.device)
        
        with torch.no_grad():
            generated_ids = model_7b.generate(**inputs, max_new_tokens=32, do_sample=False)
        generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
        output_text = processor_7b.batch_decode(generated_ids_trimmed, skip_special_tokens=True)[0]
        
        pred_task = parse_predicted_task(output_text)
        if gt_task is not None and pred_task == gt_task: correct_count += 1
        total_eval += 1
        torch.cuda.empty_cache()

    elapsed = time.time() - start_t
    peak_vram = torch.cuda.max_memory_allocated() / (1024**3)
    print_clean_summary("Qwen2-VL-7B-Instruct (4-bit)", "Dynamic Native Video (Bình thường)", total_eval, correct_count, elapsed, peak_vram)


preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

qwen-vl-utils using torchcodec to read video.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



📊 BÁO CÁO KẾT QUẢ ĐÁNH GIÁ CHÍNH THỨC:
   • Mô hình (Model)              : Qwen2-VL-7B-Instruct (4-bit)
   • Chế độ chạy (Test Mode)      : Dynamic Native Video (Bình thường)
   • Số câu dự đoán (Evaluated)   : 546 videos
   • Số câu trả lời ĐÚNG         : 82 videos
   • Độ chính xác (Accuracy)       : 15.02%
   • Thời gian thực thi (Elapsed) : 1009.82 giây (16.83 phút)
   • Peak GPU VRAM Tiêu Thụ        : 13.46 GB VRAM



In [4]:
# Cell 5: Test Qwen2-VL-7B (Mode Đóng Băng Frame 16x - Tối ưu chống OOM VRAM)
import torch, time, os, gc
from pathlib import Path
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

video_files = get_hatrec_video_files()
if not video_files:
    print("❌ KHÔNG TÌM THẤY VIDEO!")
else:
    model_id_7b = "Qwen/Qwen2-VL-7B-Instruct"
    
    # Nếu chưa load mô hình ở cell trước thì tự động load
    if 'model_7b' not in locals():
        bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4", llm_int8_enable_fp32_cpu_offload=True)
        max_memory = {0: "12GB"}
        if torch.cuda.device_count() > 1: max_memory[1] = "12GB"
        processor_7b = AutoProcessor.from_pretrained(model_id_7b)
        model_7b = Qwen2VLForConditionalGeneration.from_pretrained(model_id_7b, quantization_config=bnb_config, device_map="auto", max_memory=max_memory, offload_folder="/tmp/offload_7b")
        model_7b.eval()

    torch.cuda.reset_peak_memory_stats()
    start_t = time.time()

    prompt_text = "Watch this video carefully and classify the exact assembly action being performed into one of 7 choices: Task 0: Assembling the spring, Task 1: Placing white plastic, Task 2: Screwing-1, Task 3: Inflating valve, Task 4: Placing black plastic, Task 5: Screwing-2, Task 6: Fixing cable. State choice as: 'Task X: [Task Name]'"
    temp_static_video_7b = "/tmp/static_test_7b.mp4"
    correct_count, total_eval = 0, 0

    for video_file in video_files:
        gt_task = parse_ground_truth(str(video_file))
        eval_video = create_static_frame_video(str(video_file), temp_output_path=temp_static_video_7b, num_frames=16)
        if not eval_video: continue
        
        messages = [{"role": "user", "content": [{"type": "video", "video": eval_video, "max_pixels": 360*420, "fps": 1.0}, {"type": "text", "text": prompt_text}]}]
        text = processor_7b.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor_7b(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to(model_7b.device)
        
        with torch.no_grad():
            generated_ids = model_7b.generate(**inputs, max_new_tokens=32, do_sample=False)
        generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
        output_text = processor_7b.batch_decode(generated_ids_trimmed, skip_special_tokens=True)[0]
        
        pred_task = parse_predicted_task(output_text)
        if gt_task is not None and pred_task == gt_task: correct_count += 1
        total_eval += 1
        torch.cuda.empty_cache()

    elapsed = time.time() - start_t
    peak_vram = torch.cuda.max_memory_allocated() / (1024**3)
    print_clean_summary("Qwen2-VL-7B-Instruct (4-bit)", "Static-Frame Test (Đóng Băng Frame)", total_eval, correct_count, elapsed, peak_vram)



📊 BÁO CÁO KẾT QUẢ ĐÁNH GIÁ CHÍNH THỨC:
   • Mô hình (Model)              : Qwen2-VL-7B-Instruct (4-bit)
   • Chế độ chạy (Test Mode)      : Static-Frame Test (Đóng Băng Frame)
   • Số câu dự đoán (Evaluated)   : 546 videos
   • Số câu trả lời ĐÚNG         : 61 videos
   • Độ chính xác (Accuracy)       : 11.17%
   • Thời gian thực thi (Elapsed) : 2246.84 giây (37.45 phút)
   • Peak GPU VRAM Tiêu Thụ        : 6.68 GB VRAM



In [14]:
# Cell 6: Test Public LLaVA-1.5-7B (Mode Bình Thường - HuggingFace Native)
import torch, time, os, gc, cv2
import numpy as np
from pathlib import Path
from transformers import LlavaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

# 1. Dọn dẹp triệt để VRAM cũ
for var_name in list(globals().keys()):
    if 'model' in var_name or 'processor' in var_name or 'tokenizer' in var_name:
        try: del globals()[var_name]
        except: pass

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

video_files = get_hatrec_video_files()
if not video_files:
    print("❌ KHÔNG TÌM THẤY VIDEO!")
else:
    model_hf_id = "llava-hf/llava-1.5-7b-hf"
    model_name_display = f"HuggingFace {model_hf_id} (Quantized 4-bit NF4)"

    torch.cuda.reset_peak_memory_stats()
    start_t = time.time()

    prompt_text = "Watch this video carefully and classify the exact assembly action being performed into one of 7 choices: Task 0: Assembling the spring, Task 1: Placing white plastic, Task 2: Screwing-1, Task 3: Inflating valve, Task 4: Placing black plastic, Task 5: Screwing-2, Task 6: Fixing cable. State choice as: 'Task X: [Task Name]'"

    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
    
    # Nạp LLaVA-1.5 nguyên bản của HuggingFace
    processor_llava = AutoProcessor.from_pretrained(model_hf_id)
    model_llava = LlavaForConditionalGeneration.from_pretrained(
        model_hf_id,
        quantization_config=bnb_config,
        device_map="auto"
    )
    model_llava.eval()

    correct_count, total_eval = 0, 0
    for video_file in video_files:
        gt_task = parse_ground_truth(str(video_file))
        
        # Trích xuất Frame 0 từ video
        cap = cv2.VideoCapture(str(video_file))
        ret, frame0 = cap.read()
        cap.release()
        if not ret or frame0 is None: continue
        frame0_rgb = cv2.cvtColor(cv2.resize(frame0, (336, 336)), cv2.COLOR_BGR2RGB)

        prompt = f"USER: <image>\n{prompt_text}\nASSISTANT:"
        inputs = processor_llava(text=prompt, images=frame0_rgb, return_tensors="pt").to(model_llava.device)
        
        with torch.no_grad():
            generated_ids = model_llava.generate(**inputs, max_new_tokens=32, do_sample=False)
        output_text = processor_llava.batch_decode(generated_ids, skip_special_tokens=True)[0]
        
        pred_task = parse_predicted_task(output_text)
        if gt_task is not None and pred_task == gt_task: correct_count += 1
        total_eval += 1
        torch.cuda.empty_cache()

    elapsed = time.time() - start_t
    peak_vram = torch.cuda.max_memory_allocated() / (1024**3)
    print_clean_summary(model_name_display, "Dynamic Native Video (Bình thường)", total_eval, correct_count, elapsed, peak_vram)

    # 2. Xả 100% VRAM ở cuối Cell
    del model_llava, processor_llava
    gc.collect()
    torch.cuda.empty_cache()


processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]


📊 BÁO CÁO KẾT QUẢ ĐÁNH GIÁ CHÍNH THỨC:
   • Mô hình (Model)              : HuggingFace llava-hf/llava-1.5-7b-hf (Quantized 4-bit NF4)
   • Chế độ chạy (Test Mode)      : Dynamic Native Video (Bình thường)
   • Số câu dự đoán (Evaluated)   : 546 videos
   • Số câu trả lời ĐÚNG         : 78 videos
   • Độ chính xác (Accuracy)       : 14.29%
   • Thời gian thực thi (Elapsed) : 730.80 giây (12.18 phút)
   • Peak GPU VRAM Tiêu Thụ        : 12.01 GB VRAM



In [6]:
# Cell 6: Test Public LLaVA-1.5-7B (Mode 1: Bình Thường - Dynamic Native Video)
import torch, time, os, gc, cv2
import numpy as np
from pathlib import Path
from transformers import LlavaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

# 1. Dọn dẹp triệt để VRAM cũ
for var_name in list(globals().keys()):
    if 'model' in var_name or 'processor' in var_name or 'pipe' in var_name or 'tokenizer' in var_name:
        try: del globals()[var_name]
        except: pass

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

video_files = get_hatrec_video_files()
if not video_files:
    print("❌ KHÔNG TÌM THẤY VIDEO!")
else:
    model_hf_id = "llava-hf/llava-1.5-7b-hf"
    model_name_display = f"LLaVA-1.5-7B ({model_hf_id} - 4-bit)"

    torch.cuda.reset_peak_memory_stats()
    start_t = time.time()

    prompt_text = "Watch this video carefully and classify the exact assembly action being performed into one of 7 choices: Task 0: Assembling the spring, Task 1: Placing white plastic, Task 2: Screwing-1, Task 3: Inflating valve, Task 4: Placing black plastic, Task 5: Screwing-2, Task 6: Fixing cable. State choice as: 'Task X: [Task Name]'"

    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
    
    # Nạp LLaVA-1.5-7B nguyên bản của HuggingFace
    processor_llava = AutoProcessor.from_pretrained(model_hf_id)
    model_llava = LlavaForConditionalGeneration.from_pretrained(
        model_hf_id,
        quantization_config=bnb_config,
        device_map="auto"
    )
    model_llava.eval()

    correct_count, total_eval = 0, 0
    for video_file in video_files:
        gt_task = parse_ground_truth(str(video_file))
        
        # Trích xuất Frame 0 từ video
        cap = cv2.VideoCapture(str(video_file))
        ret, frame0 = cap.read()
        cap.release()
        if not ret or frame0 is None: continue
        frame0_rgb = cv2.cvtColor(cv2.resize(frame0, (336, 336)), cv2.COLOR_BGR2RGB)

        prompt = f"USER: <image>\n{prompt_text}\nASSISTANT:"
        inputs = processor_llava(text=prompt, images=frame0_rgb, return_tensors="pt").to(model_llava.device)
        
        with torch.no_grad():
            generated_ids = model_llava.generate(**inputs, max_new_tokens=32, do_sample=False)
        output_text = processor_llava.batch_decode(generated_ids, skip_special_tokens=True)[0]
        
        pred_task = parse_predicted_task(output_text)
        if gt_task is not None and pred_task == gt_task: correct_count += 1
        total_eval += 1
        torch.cuda.empty_cache()

    elapsed = time.time() - start_t
    peak_vram = torch.cuda.max_memory_allocated() / (1024**3)
    print_clean_summary(model_name_display, "Dynamic Native Video (Bình thường)", total_eval, correct_count, elapsed, peak_vram)

    # 2. Xả 100% VRAM ở cuối Cell
    del model_llava, processor_llava
    gc.collect()
    torch.cuda.empty_cache()


processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]


📊 BÁO CÁO KẾT QUẢ ĐÁNH GIÁ CHÍNH THỨC:
   • Mô hình (Model)              : LLaVA-1.5-7B (llava-hf/llava-1.5-7b-hf - 4-bit)
   • Chế độ chạy (Test Mode)      : Dynamic Native Video (Bình thường)
   • Số câu dự đoán (Evaluated)   : 546 videos
   • Số câu trả lời ĐÚNG         : 78 videos
   • Độ chính xác (Accuracy)       : 14.29%
   • Thời gian thực thi (Elapsed) : 896.76 giây (14.95 phút)
   • Peak GPU VRAM Tiêu Thụ        : 12.35 GB VRAM



In [7]:
# Cell 7: Test Public LLaVA-1.5-7B (Mode Đóng Băng Frame 16x - HuggingFace Native)
import torch, time, os, gc, cv2
import numpy as np
from pathlib import Path
from transformers import LlavaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

# 1. Dọn dẹp triệt để VRAM cũ
for var_name in list(globals().keys()):
    if 'model' in var_name or 'processor' in var_name or 'tokenizer' in var_name:
        try: del globals()[var_name]
        except: pass

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

video_files = get_hatrec_video_files()
if not video_files:
    print("❌ KHÔNG TÌM THẤY VIDEO!")
else:
    model_hf_id = "llava-hf/llava-1.5-7b-hf"
    model_name_display = f"HuggingFace {model_hf_id} (Quantized 4-bit NF4)"

    torch.cuda.reset_peak_memory_stats()
    start_t = time.time()

    prompt_text = "Watch this video carefully and classify the exact assembly action being performed into one of 7 choices: Task 0: Assembling the spring, Task 1: Placing white plastic, Task 2: Screwing-1, Task 3: Inflating valve, Task 4: Placing black plastic, Task 5: Screwing-2, Task 6: Fixing cable. State choice as: 'Task X: [Task Name]'"

    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
    
    processor_llava = AutoProcessor.from_pretrained(model_hf_id)
    model_llava = LlavaForConditionalGeneration.from_pretrained(
        model_hf_id,
        quantization_config=bnb_config,
        device_map="auto"
    )
    model_llava.eval()

    temp_static_video = "/tmp/static_test_llava_hf.mp4"
    correct_count, total_eval = 0, 0

    for video_file in video_files:
        gt_task = parse_ground_truth(str(video_file))
        eval_video = create_static_frame_video(str(video_file), temp_output_path=temp_static_video, num_frames=16)
        if not eval_video: continue
        
        cap = cv2.VideoCapture(eval_video)
        ret, frame0 = cap.read()
        cap.release()
        if not ret or frame0 is None: continue
        frame0_rgb = cv2.cvtColor(cv2.resize(frame0, (336, 336)), cv2.COLOR_BGR2RGB)

        prompt = f"USER: <image>\n{prompt_text}\nASSISTANT:"
        inputs = processor_llava(text=prompt, images=frame0_rgb, return_tensors="pt").to(model_llava.device)
        
        with torch.no_grad():
            generated_ids = model_llava.generate(**inputs, max_new_tokens=32, do_sample=False)
        output_text = processor_llava.batch_decode(generated_ids, skip_special_tokens=True)[0]
        
        pred_task = parse_predicted_task(output_text)
        if gt_task is not None and pred_task == gt_task: correct_count += 1
        total_eval += 1
        torch.cuda.empty_cache()

    elapsed = time.time() - start_t
    peak_vram = torch.cuda.max_memory_allocated() / (1024**3)
    print_clean_summary(model_name_display, "Static-Frame Test (Đóng Băng Frame)", total_eval, correct_count, elapsed, peak_vram)

    # 2. Xả 100% VRAM ở cuối Cell
    del model_llava, processor_llava
    gc.collect()
    torch.cuda.empty_cache()


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]


📊 BÁO CÁO KẾT QUẢ ĐÁNH GIÁ CHÍNH THỨC:
   • Mô hình (Model)              : HuggingFace llava-hf/llava-1.5-7b-hf (Quantized 4-bit NF4)
   • Chế độ chạy (Test Mode)      : Static-Frame Test (Đóng Băng Frame)
   • Số câu dự đoán (Evaluated)   : 546 videos
   • Số câu trả lời ĐÚNG         : 78 videos
   • Độ chính xác (Accuracy)       : 14.29%
   • Thời gian thực thi (Elapsed) : 824.13 giây (13.74 phút)
   • Peak GPU VRAM Tiêu Thụ        : 12.74 GB VRAM

